# cheaseBS at the campaign box edges

Does reshaping the profiles change how hard cheaseBS has to work?

Four solves per discharge: `Te_ped_scale` and `ne_ped_scale` at the frozen box
edges 0.7 and 1.3. Needs CHEASE, so this runs on NERSC.

**The open question.** The 2026-08-18 five-point 132543 run had four points
converging in 2 iterations and `ne` x1.3 needing 19, with no explanation. These
runs say whether that split follows the direction (up hard, down easy), the
variable (density hard, temperature easy), the discharge, or nothing
reproducible. The answer sets `max_iter` for the campaign, since a cap read off
the typical point truncates the hard one.

`RUN` in the setup cell gates each discharge, so you can solve one, read it, and
turn the next on. Every row carries the `DischargePhysics` for both the scaled
input (`phys_in`, frozen geometry) and the reconstruction (`phys_out`), so the
objects are there to inspect, not just the figures.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import pathlib, sys
import matplotlib.pyplot as plt

# Same root-walk the fit and scaling notebooks use, so this runs from any cwd.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pedestal_scan.py").exists())
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "reshape_convergence"))

from pedestal_scan import Campaign
import reshape_helpers as rh

# ---- switches -----------------------------------------------------------
# Solve one discharge, read it, then turn the next one on. Each costs four
# cheaseBS solves, so nothing here runs by accident.
RUN = {132543: True, 132588: False, 129015: False, 129038: False}
PLOT = True          # draw the gfile comparison after each solve
XLIM = None          # e.g. dict(rho_tor=[0.7, 1.0]) for a pedestal zoom

ROWS = {}            # shot -> rows, filled in by the cells below

camp = Campaign()
print("shots  ", camp.shots)
print("axes   ", rh.AXES)
print("scales ", rh.SCALES)
print("run    ", {k: v for k, v in RUN.items() if v} or "nothing enabled")

## Reading the output

**Table.** `dq_max` / `dq_at` / `dq_rms` / `dp_max` are the equilibrium change
over the **whole** profile and the radius where it peaks — cheaseBS reshapes all
of it, so a number read at two radii cannot say whether the reshape reached the
equilibrium. `q_err@x0` is the separate question of whether the campaign's
acceptance gate would take the point, scored at the two GENE analysis radii by
frozen policy (2026-08-16). Keep the two apart.

**Iterations.** Does 1.30 cost more than 0.70? Does `ne` cost more than `Te`?
Does it repeat across discharges?

**`accepted`.** An `R` or a `RAISED` inside the frozen box is a campaign
blocker, not a note.

**`Ip_err`.** Compare +delta against −delta on one axis. A one-sided response is
the open asymmetry.

**Figure.** A reconstruction lying on the black source curve means the reshape
never reached the equilibrium.

In [ ]:
def compare_gfiles(shot, rows, xlim=None):
    """Source EFIT against each reconstruction, straight off the objects.

    Every curve is drawn by DischargePhysics.plot_gfile, so the source and the
    reconstructions go through identical code -- and each row still carries the
    object, so anything plot_gfile does not show is one attribute away:

        r = ROWS[132543][0]
        r["phys_out"].ds                  # reconstructed profiles + geometry
        r["phys_out"]._tree["raw/gfile"]  # the reconstructed equilibrium
        r["phys_out"].plot()              # kinetic profiles
        r["phys_out"].plot_geometry()     # flux surfaces
        r["phys_in"]                      # scaled profiles, FROZEN geometry
    """
    xlim = xlim or {}
    fig = camp[shot].phys.plot_gfile(label="source EFIT", color="k", **xlim)
    colors = [plt.matplotlib.colors.to_hex(c)
              for c in plt.cm.coolwarm([0.0, 0.33, 0.66, 1.0])]
    for r, c in zip(rows, colors):
        p = r.get("phys_out")
        if p is None:
            continue
        fig = p.plot_gfile(fig=fig, color=c, **xlim,
                           label=f"{r['axis'].replace('_ped_scale','')} {r['scale']:.2f}")
    fig.suptitle(f"{shot} — source vs cheaseBS reconstructions", y=1.02)
    fig.tight_layout()
    return fig

## 132543

ELMy, low triangularity, and the shot the 2-vs-19 split came from. **Start here.**

In [ ]:
if RUN[132543]:
    ROWS[132543], wd = rh.run_bounds(camp, 132543)
    display(rh.table(ROWS[132543], shot=132543))
else:
    print("132543 disabled — set RUN[132543] = True")

In [ ]:
if RUN[132543] and PLOT:
    fig = compare_gfiles(132543, ROWS[132543], XLIM)

## 132588

ELM-free, high triangularity. Its `ne` fit is pinned on the `b_pos` bound, so treat its `ne_ped_scale` rows as suspect until that is resolved.

In [ ]:
if RUN[132588]:
    ROWS[132588], wd = rh.run_bounds(camp, 132588)
    display(rh.table(ROWS[132588], shot=132588))
else:
    print("132588 disabled — set RUN[132588] = True")

In [ ]:
if RUN[132588] and PLOT:
    fig = compare_gfiles(132588, ROWS[132588], XLIM)

## 129015

ELMy, low triangularity.

In [ ]:
if RUN[129015]:
    ROWS[129015], wd = rh.run_bounds(camp, 129015)
    display(rh.table(ROWS[129015], shot=129015))
else:
    print("129015 disabled — set RUN[129015] = True")

In [ ]:
if RUN[129015] and PLOT:
    fig = compare_gfiles(129015, ROWS[129015], XLIM)

## 129038

ELM-free, low triangularity. Weakest fit of the four and a placeholder analysis radius, so its gate verdict carries less weight than the other three.

In [ ]:
if RUN[129038]:
    ROWS[129038], wd = rh.run_bounds(camp, 129038)
    display(rh.table(ROWS[129038], shot=129038))
else:
    print("129038 disabled — set RUN[129038] = True")

In [ ]:
if RUN[129038] and PLOT:
    fig = compare_gfiles(129038, ROWS[129038], XLIM)

## All solved discharges together

In [ ]:
import pandas as pd
if ROWS:
    display(pd.DataFrame(
        [{"shot": s, "axis": r["axis"], "scale": r["scale"],
          "iters": r.get("iterations"), "accepted": r.get("accepted"),
          "dq_max": r.get("dq_max"), "wall_s": r.get("wall_s")}
         for s, rs in ROWS.items() for r in rs]
    ).pivot_table(index=["shot", "axis"], columns="scale", values="iters",
                  dropna=False))
else:
    print("nothing solved yet")